# 13 — End-to-end baselineの固定

改造前の設定と評価項目を固定し、比較可能な実験単位を作ります。

**前提**: `12_force_to_joint_torque.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## Baseline契約

比較ごとに最低限、commit/tree、robot、scene、seed、速度指令、摩擦、
gait、MPC type、N、dt、Q/R、solver statusを保存します。
`config.py` のdisk既定sceneは `perlin` なので、平地試験は `flat` を明示します。

In [2]:
from quadruped_pympc import config as cfg
import json
record = {
    "robot": cfg.robot,
    "scene_for_experiment": "flat",
    "seed": 0,
    "command_mps": [0.2, 0.0, 0.0],
    "ground_mu": 0.8,
    "mpc_mu": cfg.mpc_params["mu"],
    "gait": cfg.simulation_params["gait"],
    "step_freq": cfg.simulation_params["gait_params"]["trot"]["step_freq"],
    "duty_factor": cfg.simulation_params["gait_params"]["trot"]["duty_factor"],
    "horizon": cfg.mpc_params["horizon"],
    "mpc_dt": cfg.mpc_params["dt"],
}
print(json.dumps(record, indent=2))

{
  "robot": "go2",
  "scene_for_experiment": "flat",
  "seed": 0,
  "command_mps": [
    0.2,
    0.0,
    0.0
  ],
  "ground_mu": 0.8,
  "mpc_mu": 0.42,
  "gait": "trot",
  "step_freq": 1.35,
  "duty_factor": 0.74,
  "horizon": 12,
  "mpc_dt": 0.02
}


## 重い実行

次のセルは意図的に既定OFFです。全身NMPCはsolver生成とMuJoCo計算を伴います。
実行するときだけ `RUN_HEAVY=True` にし、Jupyterを
`.env.workshop` source後に起動してください。

In [3]:
RUN_HEAVY = False
if RUN_HEAVY:
    from quadruped_pympc import config as cfg
    from simulation.simulation import run_simulation
    old_scene = cfg.simulation_params["scene"]
    cfg.simulation_params["scene"] = "flat"
    try:
        run_simulation(
            cfg, num_episodes=1, num_seconds_per_episode=2,
            ref_base_lin_vel=(0.2, 0.2), ref_base_ang_vel=(0., 0.),
            friction_coeff=(0.8, 0.8), base_vel_command_type="forward",
            seed=0, render=False,
        )
    finally:
        cfg.simulation_params["scene"] = old_scene
else:
    print("Skipped. Set RUN_HEAVY=True for a 2 s headless baseline.")

Skipped. Set RUN_HEAVY=True for a 2 s headless baseline.


成功判定は転倒しないことだけでは不十分です。速度RMSE、姿勢RMS/最大値、
高さ誤差、摩擦margin、トルク飽和率、solver失敗率、計算時間を同じ時間窓で比較します。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。